In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 21. RLHF

> **学習目標**
> - 学習 (MDP, , , ) LLM
> - (Policy Gradient) REINFORCE 複雑度
> - PPO clipped objective
> - モデル(RM) 学習

## 21.1 SFT 問題

SFT モデル " " :
- ****
- 複雑度
- "(alignment)" : モデル

## 21.2 RLHF 3

1. **SFT**: (Ch 20)
2. **Reward Model (RM)**: モデル 学習
3. **PPO**: RM (=LLM) 学習

## 21.3 学習 — MDP

**MDP** (Markov Decision Process): $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$
- $\mathcal{S}$: 空間
- $\mathcal{A}$: 空間
- $P(s'|s, a)$:
- $R(s, a)$:
- $\gamma$:

LLM :
- $s_t$ = $[w_0, \ldots, w_{t-1}]$
- $a_t$ = $w_t$
- $\pi_\theta(a_t | s_t) = P(w_t | w_{<t}; \theta)$
- $R$ = RM ( )


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

#   (toy)
vocab_size = 100  #    
print(f"vocab_size (toy): {vocab_size}")

## 21.4 (REINFORCE)

: $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$ .

$$\nabla J(\theta) = \mathbb{E}_{\tau} \left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

- $G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$:
- $\nabla \log \pi$: → " "
- $G_t$ →

**Advantage** $A_t = G_t - V(s_t)$: $V$ .


In [ ]:
# REINFORCE  ( toy MDP)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# :  bandit (3 ,  分布 )
true_rewards = [0.2, 0.5, 0.8]  #  2 

class PolicyNet(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(1, 16)
        self.head = nn.Linear(16, n_actions)
    def forward(self, x):
        return self.head(F.relu(self.fc(x)))

policy = PolicyNet(3)
opt = torch.optim.Adam(policy.parameters(), lr=0.01)

n_episodes = 500
all_rewards = []
for ep in range(n_episodes):
    # : 1 step (bandit)
    state = torch.tensor([[1.0]])
    logits = policy(state)
    probs = F.softmax(logits, dim=-1)
    action = torch.multinomial(probs, 1).item()
    reward = true_rewards[action] + np.random.randn() * 0.05

    # REINFORCE: log_prob * reward
    log_prob = F.log_softmax(logits, dim=-1)[0, action]
    loss = -log_prob * reward  #  →  
    opt.zero_grad()
    loss.backward()
    opt.step()
    all_rewards.append(reward)

print(f" 50  Mean Reward: {np.mean(all_rewards[:50]):.3f}")
print(f" 50  Mean Reward: {np.mean(all_rewards[-50:]):.3f}")
print(f"Theory Maximum: {max(true_rewards):.3f}")


## 21.5 PPO — Trust Region Clipped Objective

REINFORCE . PPO .

****: $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$

**Clipped Objective**:
$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[\min\left(r_t \hat{A}_t, \, \mathrm{clip}(r_t, 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

- $\epsilon \approx 0.2$:
- min clip →
- trust region


In [ ]:
# PPO  (toy 問題)
import torch
import torch.nn as nn
import torch.nn.functional as F

class Policy(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(4, 32)
        self.head = nn.Linear(32, n_actions)
    def forward(self, x):
        return self.head(F.tanh(self.fc(x)))

#   ()
def env_step(state, action):
    reward = float((state.sum() + action - 2) / 10)  #  
    next_state = state + torch.randn(4) * 0.1
    return next_state, reward, False

policy = Policy(3)
opt = torch.optim.Adam(policy.parameters(), lr=3e-4)

# PPO 
def ppo_update(policy, opt, states, actions, old_log_probs, advantages, returns, values,
               epsilon=0.2, n_epochs=4):
    for _ in range(n_epochs):
        logits = policy(states)
        new_log_probs = F.log_softmax(logits, dim=-1)
        new_log_probs_a = new_log_probs.gather(1, actions.unsqueeze(1)).squeeze()

        ratio = torch.exp(new_log_probs_a - old_log_probs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()

        #  関数  ()
        critic_loss = F.mse_loss(values.squeeze(), returns)

        loss = actor_loss + 0.5 * critic_loss
        opt.zero_grad()
        loss.backward()
        opt.step()

# 学習 
n_iterations = 50
all_rewards = []
for it in range(n_iterations):
    # rollout 
    states, actions, rewards, old_log_probs = [], [], [], []
    state = torch.randn(1, 4)
    ep_reward = 0
    for step in range(20):
        with torch.no_grad():
            logits = policy(state)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            old_log_prob = dist.log_prob(action)
        next_state, reward, done = env_step(state.squeeze(), action.item())
        states.append(state.squeeze())
        actions.append(action)
        rewards.append(reward)
        old_log_probs.append(old_log_prob)
        ep_reward += reward
        state = next_state.unsqueeze(0)
    all_rewards.append(ep_reward / 20)

    # advantage 計算 (:  )
    advantages = torch.tensor(rewards)
    returns = torch.tensor(rewards)
    values = torch.zeros_like(returns)  #   ()

    ppo_update(policy, opt, torch.stack(states), torch.tensor(actions),
               torch.stack(old_log_probs), advantages, returns, values)

print(f" 10  Mean Reward: {np.mean(all_rewards[:10]):.4f}")
print(f" 10  Mean Reward: {np.mean(all_rewards[-10:]):.4f}")


## 21.6 モデル (RM) 学習

RM モデル. ** データ** (chosen vs rejected) 学習.

**Bradley-Terry モデル**:
$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

 (NLL):
$$\mathcal{L}_{\text{RM}} = -\log \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

- $\mathbf{y}_w$: (chosen)
- $\mathbf{y}_l$: (rejected)
- $r(\mathbf{x}, \mathbf{y})$: RM

RM : SFT モデル 出力 .


In [ ]:
# RM 学習 
import torch
import torch.nn as nn
import torch.nn.functional as F

class RewardModel(nn.Module):
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.rnn = nn.LSTM(d_model, d_model, batch_first=True)
        self.head = nn.Linear(d_model, 1)  #  

    def forward(self, x):
        emb = self.emb(x)
        _, (h, _) = self.rnn(emb)
        return self.head(h.squeeze(0))  # (B,)

# toy  データ
n_samples = 100
torch.manual_seed(0)
rm = RewardModel(vocab_size=vocab_size, d_model=32)
opt = torch.optim.AdamW(rm.parameters(), lr=1e-3)

#  chosen/rejected pairs ()
print("RM Training :")
losses = []
for step in range(200):
    #  データ:  
    chosen = torch.randint(0, vocab_size, (8, 16))
    rejected = torch.randint(0, vocab_size, (8, 16))
    #  : chosen  ID   
    # ( RM 学習 )
    r_chosen = rm(chosen).squeeze(-1)
    r_rejected = rm(rejected).squeeze(-1)
    loss = -F.logsigmoid(r_chosen - r_rejected).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('RM Loss')
plt.title('Reward Model Training (Bradley-Terry)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch21_rm_training.png', dpi=100, bbox_inches='tight')
plt.show()
print(f" RM loss: {losses[-1]:.4f} (Theory Minimum ≈ 0.69 = -log(0.5) )")


## 21.7 KL — モデル 複雑度

RLHF RM モデル (reward hacking).

: モデル(SFT モデル) KL .

$$R_{\text{total}} = R_{\text{RM}}(\mathbf{x}, \mathbf{y}) - \beta \, D_{\text{KL}}(\pi_\theta(\cdot|\mathbf{x}) \| \pi_{\text{ref}}(\cdot|\mathbf{x}))$$

- $\beta$: KL 複雑度 ( 0.01~0.5)
- $\pi_{\text{ref}}$: SFT モデル 
- KL: $\sum_t \log \frac{\pi_\theta(w_t|...)}{\pi_{\text{ref}}(w_t|...)}$


In [ ]:
# KL  計算 ( vs )
def token_kl_div(policy_logits, ref_logits):
    """ KL Divergence."""
    p = F.softmax(policy_logits, dim=-1)
    log_p = F.log_softmax(policy_logits, dim=-1)
    log_q = F.log_softmax(ref_logits, dim=-1)
    return (p * (log_p - log_q)).sum(dim=-1)  # (B, T)

# 
torch.manual_seed(0)
B, T, V = 4, 16, 100
ref_logits = torch.randn(B, T, V)
policy_logits = ref_logits + torch.randn(B, T, V) * 0.5  #   Policy

kl = token_kl_div(policy_logits, ref_logits)
print(f"KL  shape: {kl.shape}")
print(f"Mean KL: {kl.mean():.4f}")
print(f"Maximum KL: {kl.max():.4f}")
print("\n=> Policy   KL Increase. RLHF   .")


## 21.8 [CPU/GPU ベンチマーク ⑨] SFT vs RLHF 学習 比較

RLHF モデル 4 : , , RM, 関数. ·時間 3~4.


In [ ]:
# RLHF   ()
print("RLHF Training  Model:")
print("  1. Policy model (Training)")
print("  2. Reference model (, KL)")
print("  3. Reward model ()")
print("  4. Value/Critic model (Training, optional)")
print()

# : 7B LLaMA RLHF
model_size_gb = 7 * 2 * 4  # 7B params * 2 bytes (FP16) * 4 models? No:
# : policy (FP32 grad, FP32 opt state)
# AdamW: param + grad + m + v = 4
# 7B モデル FP16: 14GB
# AdamW  FP32: 7B * 8 bytes = 56GB
# policy: 14 + 56 = 70GB
# reference: 14GB
# RM: 14GB
# value: 14 + 56 = 70GB
# :  168GB → A100 80GB 2 

print("7B Model RLHF Memory Estimation:")
print(f"  Policy (FP16 + AdamW FP32): ~70GB")
print(f"  Reference (FP16, ): ~14GB")
print(f"  Reward Model (FP16, ): ~14GB")
print(f"  Value (FP16 + AdamW): ~70GB")
print(f"  : ~168GB (A100 80GB × 2 )")
print("\n=> RLHF  . DPO(Ch 22)  Efficiency .")


## 21.9 RLHF

- ** (reward hacking)**: RM
- ** (mode collapse)**:
- ****: PPO

## 21.10 要点

| | | |
|---|---|---|
| | $\nabla J = \mathbb{E}[\nabla\log\pi \cdot G]$ | REINFORCE |
| Advantage | $A = G - V$ | |
| PPO clip | $\min(rA, \text{clip}(r)A)$ | trust region |
| RM | $-\log\sigma(r_w - r_l)$ | Bradley-Terry |
| KL | $R - \beta D_{KL}$ | |

## 演習問題

1. REINFORCE CartPole toy 学習.
2. PPO $\epsilon = 0.1, 0.2, 0.3$ 比較 .
3. RM 学習 chosen rejected vs loss 比較.
4. KL $\beta = 0, 0.1, 1.0$ .
5. RLHF DPO .

> 解答: `solutions/ch21_solutions.ipynb`
